# 📊 Кредитный рейтинг

Этот ноутбук позволяет:
- Выбрать компанию и методологию рейтинга
- Рассчитать Base Rating (рейтинг на основе истории)
- Рассчитать Forecast Rating (рейтинг на основе прогноза)
- Рассчитать Stress Rating (рейтинг на основе стресс-сценариев)
- Просмотреть анализ миграции рейтинга
- Просмотреть анализ устойчивости рейтинга

## 📋 Этапы работы:

1. **Загрузка данных**: Загрузка базовых данных компании
2. **Выбор методологии**: Выбор методологии рейтинга (S&P, Moody's, Fitch)
3. **Base Rating**: Расчет рейтинга на основе истории
4. **Forecast Rating**: Расчет рейтинга на основе прогноза
5. **Stress Rating**: Расчет рейтинга для стресс-сценариев
6. **Анализ**: Анализ миграции и устойчивости рейтинга

## 📚 Документация

- `docs/rating/RATING_GUIDE.md` - Полное руководство
- `configs/rating_config.yaml` - Конфигурация рейтинговых шкал

In [ ]:
# ── Setup ──────────────────────────────────────────────
import sys, pathlib
ROOT = pathlib.Path('.').resolve()
for _p in [ROOT] + list(ROOT.parents):
    if (_p / 'engine').is_dir():
        ROOT = _p; break
sys.path.insert(0, str(ROOT))

COMPANY = None
for _p in [pathlib.Path('.').resolve()] + list(pathlib.Path('.').resolve().parents):
    if _p.parent.name == 'companies':
        COMPANY = _p.name; break
if not COMPANY: COMPANY = 'nornickel'  # fallback
print(f'Company: {COMPANY}, ROOT: {ROOT}')


In [ ]:
# Импорты и настройка
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while ROOT.parent != ROOT and not (ROOT / 'engine').exists():
    ROOT = ROOT.parent
if not (ROOT / 'engine').exists():
    raise RuntimeError(f"Can't locate project root from {Path.cwd()}")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print(f"📁 ROOT: {ROOT}")

In [ ]:
# Импорты модулей
import pandas as pd
import numpy as np
from pathlib import Path
from typing import Optional, List
from IPython.display import display, Markdown, HTML, clear_output
import ipywidgets as widgets
from ipywidgets import HBox, VBox, Layout, Output

# Импорты движка
from engine.database.data_mart import get_data_mart
from engine.rating.rating_runner import RatingRunner
from engine.rating.rating_models import RatingMethodology

print("✅ Модули загружены")

## 1️⃣ Выбор компании и методологии

## 4️⃣ Визуализация результатов

Визуализация рейтингов и метрик для анализа.

In [ ]:
# Визуализация результатов рейтинга
import matplotlib.pyplot as plt
import seaborn as sns

# Настройка стиля
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

def plot_rating_timeline():
    """Визуализация изменения рейтинга во времени"""
    if 'forecast' not in rating_results:
        print("⚠️ Нет результатов Forecast Rating для визуализации.")
        return
    
    results = rating_results['forecast']
    if not results:
        return
    
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10))
    
    years = [r.year for r in results]
    ratings_numeric = [r.rating.to_numeric() for r in results]
    scores = [r.score for r in results]
    pds = [r.pd for r in results]
    
    # График рейтинга (числовое значение)
    ax1.plot(years, ratings_numeric, marker='o', linewidth=2, markersize=8, color='blue')
    ax1.set_xlabel('Год', fontsize=12)
    ax1.set_ylabel('Рейтинг (числовое значение)', fontsize=12)
    ax1.set_title('Изменение рейтинга во времени', fontsize=14, fontweight='bold')
    ax1.grid(True, alpha=0.3)
    ax1.invert_yaxis()  # Инвертируем ось Y (меньше = лучше)
    
    # Добавляем подписи рейтингов
    for i, (year, rating) in enumerate(zip(years, [r.rating.value for r in results])):
        ax1.annotate(rating, (year, ratings_numeric[i]), 
                    textcoords="offset points", xytext=(0,10), ha='center', fontsize=9)
    
    # График PD
    ax2.plot(years, pds, marker='s', linewidth=2, markersize=8, color='red')
    ax2.set_xlabel('Год', fontsize=12)
    ax2.set_ylabel('Probability of Default', fontsize=12)
    ax2.set_title('Изменение PD во времени', fontsize=14, fontweight='bold')
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

def plot_rating_metrics_comparison():
    """Сравнение метрик по категориям"""
    if 'base' not in rating_results:
        print("⚠️ Нет результатов Base Rating для визуализации.")
        return
    
    result = rating_results['base']
    metrics = result.metrics
    
    # Подготовка данных
    categories = {
        'Profitability': [
            ('ROA', metrics.profitability.roa),
            ('ROE', metrics.profitability.roe),
            ('ROIC', metrics.profitability.roic),
            ('EBITDA Margin', metrics.profitability.ebitda_margin),
        ],
        'Leverage': [
            ('Debt/Equity', metrics.leverage.debt_to_equity),
            ('Debt/EBITDA', metrics.leverage.debt_to_ebitda),
            ('Net Debt/EBITDA', metrics.leverage.net_debt_to_ebitda),
        ],
        'Coverage': [
            ('ICR', metrics.coverage.icr),
            ('DSCR', metrics.coverage.dscr),
            ('FFO/Interest', metrics.coverage.ffo_to_interest),
        ],
        'Liquidity': [
            ('Current Ratio', metrics.liquidity.current_ratio),
            ('Quick Ratio', metrics.liquidity.quick_ratio),
            ('Cash/ST Debt', metrics.liquidity.cash_to_short_term_debt),
        ]
    }
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    axes = axes.flatten()
    
    for idx, (category, metric_list) in enumerate(categories.items()):
        ax = axes[idx]
        
        names = [m[0] for m in metric_list]
        values = [m[1] for m in metric_list]
        
        bars = ax.barh(names, values, alpha=0.7)
        ax.set_xlabel('Значение', fontsize=10)
        ax.set_title(category, fontsize=12, fontweight='bold')
        ax.grid(True, alpha=0.3, axis='x')
        
        # Добавляем значения на столбцы
        for i, (name, value) in enumerate(zip(names, values)):
            ax.text(value, i, f'{value:.2f}', va='center', ha='left' if value >= 0 else 'right', fontsize=9)
    
    plt.tight_layout()
    plt.show()

def plot_stress_resilience():
    """Визуализация устойчивости рейтинга к стресс-сценариям"""
    if 'stress' not in rating_results or 'base' not in rating_results:
        print("⚠️ Нет результатов Stress Rating или Base Rating для визуализации.")
        return
    
    base_rating = rating_results['base']
    stress_ratings = rating_results['stress']
    
    if not stress_ratings:
        return
    
    # Анализ устойчивости
    runner = init_runner()
    resilience = runner.stress_calculator.analyze_rating_resilience(
        base_rating=base_rating,
        stress_ratings=stress_ratings
    )
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
    
    # График рейтингов по сценариям
    scenarios = [getattr(r, 'scenario_name', f'Scenario {i}') for i, r in enumerate(stress_ratings)]
    stress_ratings_numeric = [r.rating.to_numeric() for r in stress_ratings]
    base_numeric = base_rating.rating.to_numeric()
    
    ax1.barh(scenarios, stress_ratings_numeric, alpha=0.7, color='orange')
    ax1.axvline(x=base_numeric, color='blue', linestyle='--', linewidth=2, label=f'Base: {base_rating.rating.value}')
    ax1.set_xlabel('Рейтинг (числовое значение)', fontsize=12)
    ax1.set_title('Рейтинги по стресс-сценариям', fontsize=14, fontweight='bold')
    ax1.invert_xaxis()  # Инвертируем ось X (меньше = лучше)
    ax1.legend()
    ax1.grid(True, alpha=0.3, axis='x')
    
    # График PD по сценариям
    stress_pds = [r.pd for r in stress_ratings]
    base_pd = base_rating.pd
    
    ax2.barh(scenarios, stress_pds, alpha=0.7, color='red')
    ax2.axvline(x=base_pd, color='blue', linestyle='--', linewidth=2, label=f'Base PD: {base_pd:.4f}')
    ax2.set_xlabel('Probability of Default', fontsize=12)
    ax2.set_title('PD по стресс-сценариям', fontsize=14, fontweight='bold')
    ax2.legend()
    ax2.grid(True, alpha=0.3, axis='x')
    
    plt.tight_layout()
    plt.show()
    
    # Выводим статистику
    display(Markdown("### 🛡️ Статистика устойчивости"))
    display(Markdown(f"**Базовый рейтинг:** {resilience['base_rating']}"))
    display(Markdown(f"**Максимальный downgrade:** {resilience.get('max_downgrade_notches', 0):.1f} уровня"))
    if resilience.get('worst_case'):
        worst = resilience['worst_case']
        display(Markdown(f"**Worst case сценарий:** {worst['scenario']} (рейтинг: {worst['rating']}, downgrade: {worst['downgrade_notches']:.1f})"))
    display(Markdown(f"**Устойчивость:** {resilience['resilience_score']:.2%}"))

# Примеры использования:
# plot_rating_timeline()
# plot_rating_metrics_comparison()
# plot_stress_resilience()

In [ ]:
# Список компаний
COMPANIES = [d.name for d in (ROOT / 'companies').iterdir() 
             if d.is_dir() and not d.name.startswith('.')]

print(f"📊 Доступные компании: {COMPANIES}")

# Виджеты для выбора
company_dropdown = widgets.Dropdown(
    options=COMPANIES,
    value='nornickel' if 'nornickel' in COMPANIES else (COMPANIES[0] if COMPANIES else None),
    description='Компания:',
    style={'description_width': '150px'},
    layout=Layout(width='400px')
)

methodology_dropdown = widgets.Dropdown(
    options=[('S&P', RatingMethodology.SP), ('Moody\'s', RatingMethodology.MOODYS), ('Fitch', RatingMethodology.FITCH)],
    value=RatingMethodology.SP,
    description='Методология:',
    style={'description_width': '150px'},
    layout=Layout(width='400px')
)

sector_dropdown = widgets.Dropdown(
    options=['', 'steel', 'oil_gas', 'mining', 'retail', 'other'],
    value='steel',
    description='Сектор:',
    style={'description_width': '150px'},
    layout=Layout(width='400px')
)

base_year_text = widgets.Text(
    value='2024',
    description='Год для Base Rating:',
    style={'description_width': '150px'},
    layout=Layout(width='400px')
)

forecast_years_text = widgets.Text(
    value='2025,2026,2027,2028,2029',
    description='Годы прогноза:',
    style={'description_width': '150px'},
    layout=Layout(width='400px')
)

calculate_base_button = widgets.Button(
    description='📊 Base Rating',
    button_style='info',
    icon='chart-line'
)

calculate_forecast_button = widgets.Button(
    description='📈 Forecast Rating',
    button_style='info',
    icon='chart-bar'
)

calculate_stress_button = widgets.Button(
    description='🧪 Stress Rating',
    button_style='warning',
    icon='flask'
)

full_analysis_button = widgets.Button(
    description='🚀 Полный анализ',
    button_style='success',
    icon='rocket'
)

status_output = widgets.Output()
results_output = widgets.Output()

display(VBox([
    HBox([company_dropdown, methodology_dropdown]),
    HBox([sector_dropdown, base_year_text]),
    HBox([forecast_years_text]),
    HBox([calculate_base_button, calculate_forecast_button, calculate_stress_button, full_analysis_button]),
    status_output,
    results_output
]))

In [ ]:
# Глобальные переменные для результатов
rating_results = {}
runner = None

def parse_forecast_years(years_str: str) -> List[int]:
    """Парсинг строки с годами"""
    try:
        years = [int(y.strip()) for y in years_str.split(',')]
        return sorted(years)
    except Exception as e:
        print(f"Ошибка парсинга лет: {e}")
        return [2025, 2026, 2027, 2028, 2029]

def init_runner():
    """Инициализация RatingRunner"""
    global runner
    
    company = company_dropdown.value
    methodology = methodology_dropdown.value
    
    runner = RatingRunner(
        root=ROOT,
        company=company,
        methodology=methodology
    )
    
    return runner

def calculate_base_rating_func(b):
    """Расчет Base Rating"""
    global rating_results
    
    with status_output:
        status_output.clear_output()
        print("📊 Расчет Base Rating...")
    
    try:
        runner = init_runner()
        
        base_year = int(base_year_text.value)
        sector = sector_dropdown.value if sector_dropdown.value else None
        
        result = runner.calculate_base_rating(
            year=base_year,
            sector=sector
        )
        
        rating_results['base'] = result
        
        # Отображаем результаты
        with results_output:
            results_output.clear_output()
            display(Markdown(f"## ✅ Base Rating для {base_year}"))
            display(Markdown(f"**Рейтинг:** {result.rating.value}"))
            display(Markdown(f"**Score:** {result.score:.2f}"))
            display(Markdown(f"**PD (Probability of Default):** {result.pd:.4f}"))
            display(Markdown(f"**Методология:** {result.methodology.value.upper()}"))
            
            # Метрики
            display(Markdown("### 📈 Метрики"))
            
            prof = result.metrics.profitability
            display(Markdown("#### Profitability:"))
            display(Markdown(f"- ROA: {prof.roa:.2f}%"))
            display(Markdown(f"- ROE: {prof.roe:.2f}%"))
            display(Markdown(f"- EBITDA Margin: {prof.ebitda_margin:.2f}%"))
            
            lev = result.metrics.leverage
            display(Markdown("#### Leverage:"))
            display(Markdown(f"- Debt/Equity: {lev.debt_to_equity:.2f}"))
            display(Markdown(f"- Debt/EBITDA: {lev.debt_to_ebitda:.2f}"))
            display(Markdown(f"- Net Debt/EBITDA: {lev.net_debt_to_ebitda:.2f}"))
            
            cov = result.metrics.coverage
            display(Markdown("#### Coverage:"))
            display(Markdown(f"- ICR: {cov.icr:.2f}"))
            display(Markdown(f"- DSCR: {cov.dscr:.2f}"))
            
            liq = result.metrics.liquidity
            display(Markdown("#### Liquidity:"))
            display(Markdown(f"- Current Ratio: {liq.current_ratio:.2f}"))
            display(Markdown(f"- Quick Ratio: {liq.quick_ratio:.2f}"))
        
        with status_output:
            status_output.clear_output()
            print("✅ Base Rating рассчитан")
    
    except Exception as e:
        with status_output:
            status_output.clear_output()
            print(f"❌ Ошибка: {e}")
        import traceback
        traceback.print_exc()

def calculate_forecast_rating_func(b):
    """Расчет Forecast Rating"""
    global rating_results
    
    with status_output:
        status_output.clear_output()
        print("📈 Расчет Forecast Rating...")
    
    try:
        runner = init_runner()
        
        forecast_years = parse_forecast_years(forecast_years_text.value)
        sector = sector_dropdown.value if sector_dropdown.value else None
        
        results = runner.calculate_forecast_rating(
            forecast_years=forecast_years,
            sector=sector
        )
        
        rating_results['forecast'] = results
        
        # Анализ миграции
        migration = runner.forecast_calculator.analyze_rating_migration(results)
        
        # Отображаем результаты
        with results_output:
            results_output.clear_output()
            display(Markdown("## ✅ Forecast Rating"))
            
            # Таблица рейтингов по годам
            display(Markdown("### 📊 Рейтинги по годам"))
            rating_data = []
            for result in results:
                rating_data.append({
                    'Год': result.year,
                    'Рейтинг': result.rating.value,
                    'Score': f"{result.score:.2f}",
                    'PD': f"{result.pd:.4f}"
                })
            
            if rating_data:
                rating_df = pd.DataFrame(rating_data)
                display(rating_df)
            
            # Анализ миграции
            display(Markdown("### 📉 Анализ миграции рейтинга"))
            display(Markdown(f"**Общая миграция:** {migration['migration']}"))
            display(Markdown(f"**Из:** {migration['from']} → **В:** {migration['to']}"))
            
            if migration['changes']:
                display(Markdown("#### Изменения по годам:"))
                changes_data = []
                for change in migration['changes']:
                    changes_data.append({
                        'Год': change['year'],
                        'Из': change['from'],
                        'В': change['to'],
                        'Тип': change['type'],
                        'Изменение PD': f"{change['pd_change']:+.4f}"
                    })
                changes_df = pd.DataFrame(changes_data)
                display(changes_df)
        
        with status_output:
            status_output.clear_output()
            print("✅ Forecast Rating рассчитан")
    
    except Exception as e:
        with status_output:
            status_output.clear_output()
            print(f"❌ Ошибка: {e}")
        import traceback
        traceback.print_exc()

def calculate_stress_rating_func(b):
    """Расчет Stress Rating"""
    global rating_results
    
    with status_output:
        status_output.clear_output()
        print("🧪 Расчет Stress Rating...")
    
    try:
        runner = init_runner()
        
        # Получаем список доступных сценариев
        from engine.stress.scenarios import ScenarioLoader
        scenario_loader = ScenarioLoader()
        scenario_loader.load(ROOT)
        available_scenarios = scenario_loader.list_scenarios()
        
        if not available_scenarios:
            with results_output:
                results_output.clear_output()
                display(Markdown("⚠️ Нет доступных стресс-сценариев"))
            return
        
        forecast_years = parse_forecast_years(forecast_years_text.value)
        sector = sector_dropdown.value if sector_dropdown.value else None
        
        # Рассчитываем рейтинг для каждого сценария
        stress_results = []
        
        for scenario_name in available_scenarios[:3]:  # Первые 3 сценария
            try:
                result = runner.calculate_stress_rating(
                    scenario_name=scenario_name,
                    year=forecast_years[0] if forecast_years else 2025,
                    sector=sector
                )
                stress_results.append(result)
            except Exception as e:
                print(f"Ошибка для сценария {scenario_name}: {e}")
        
        rating_results['stress'] = stress_results
        
        # Анализ устойчивости (если есть base rating)
        if 'base' in rating_results and stress_results:
            resilience = runner.stress_calculator.analyze_rating_resilience(
                base_rating=rating_results['base'],
                stress_ratings=stress_results
            )
            
            # Отображаем результаты
            with results_output:
                results_output.clear_output()
                display(Markdown("## ✅ Stress Rating"))
                
                display(Markdown("### 📊 Рейтинги по сценариям"))
                stress_data = []
                for result in stress_results:
                    scenario_name = getattr(result, 'scenario_name', 'unknown')
                    stress_data.append({
                        'Сценарий': scenario_name,
                        'Рейтинг': result.rating.value,
                        'Score': f"{result.score:.2f}",
                        'PD': f"{result.pd:.4f}"
                    })
                
                if stress_data:
                    stress_df = pd.DataFrame(stress_data)
                    display(stress_df)
                
                # Анализ устойчивости
                display(Markdown("### 🛡️ Анализ устойчивости рейтинга"))
                display(Markdown(f"**Базовый рейтинг:** {resilience['base_rating']}"))
                display(Markdown(f"**Базовая PD:** {resilience['base_pd']:.4f}"))
                display(Markdown(f"**Средняя PD в стресс-условиях:** {resilience['avg_stress_pd']:.4f}"))
                display(Markdown(f"**Устойчивость:** {resilience['resilience_score']:.2%}"))
                
                if resilience['downgrades']:
                    display(Markdown("#### ⚠️ Снижения рейтинга:"))
                    for downgrade in resilience['downgrades']:
                        display(Markdown(f"- {downgrade['scenario']}: {downgrade['rating']} (снижение на {downgrade['downgrade_notches']:.1f} уровня)"))
                
                if resilience['stable']:
                    display(Markdown("#### ✅ Стабильные сценарии:"))
                    for stable in resilience['stable']:
                        display(Markdown(f"- {stable['scenario']}: {stable['rating']}"))
        else:
            with results_output:
                results_output.clear_output()
                display(Markdown("## ✅ Stress Rating"))
                display(Markdown("⚠️ Для анализа устойчивости требуется Base Rating"))
        
        with status_output:
            status_output.clear_output()
            print("✅ Stress Rating рассчитан")
    
    except Exception as e:
        with status_output:
            status_output.clear_output()
            print(f"❌ Ошибка: {e}")
        import traceback
        traceback.print_exc()

def full_analysis_func(b):
    """Полный анализ рейтинга"""
    global rating_results
    
    with status_output:
        status_output.clear_output()
        print("🚀 Запуск полного анализа рейтинга...")
    
    try:
        runner = init_runner()
        
        base_year = int(base_year_text.value)
        forecast_years = parse_forecast_years(forecast_years_text.value)
        sector = sector_dropdown.value if sector_dropdown.value else None
        
        # Получаем список доступных стресс-сценариев
        from engine.stress.scenarios import ScenarioLoader
        scenario_loader = ScenarioLoader()
        scenario_loader.load(ROOT)
        available_scenarios = scenario_loader.list_scenarios()[:3]  # Первые 3
        
        analysis = runner.run_full_rating_analysis(
            base_year=base_year,
            forecast_years=forecast_years,
            stress_scenarios=available_scenarios if available_scenarios else None,
            sector=sector
        )
        
        rating_results['full_analysis'] = analysis
        
        # Отображаем результаты
        with results_output:
            results_output.clear_output()
            display(Markdown("## 🚀 Полный анализ рейтинга"))
            
            # Base Rating
            if analysis.get('base_rating'):
                base = analysis['base_rating']
                display(Markdown("### 📊 Base Rating"))
                display(Markdown(f"**Рейтинг:** {base.rating.value}"))
                display(Markdown(f"**PD:** {base.pd:.4f}"))
            
            # Forecast Rating
            if analysis.get('forecast_ratings'):
                display(Markdown("### 📈 Forecast Rating"))
                forecast_data = []
                for result in analysis['forecast_ratings']:
                    forecast_data.append({
                        'Год': result.year,
                        'Рейтинг': result.rating.value,
                        'PD': f"{result.pd:.4f}"
                    })
                forecast_df = pd.DataFrame(forecast_data)
                display(forecast_df)
                
                # Миграция
                if analysis.get('migration_analysis'):
                    migration = analysis['migration_analysis']
                    display(Markdown(f"**Миграция:** {migration['migration']} ({migration['from']} → {migration['to']})"))
            
            # Stress Rating
            if analysis.get('stress_ratings'):
                display(Markdown("### 🧪 Stress Rating"))
                stress_data = []
                for result in analysis['stress_ratings']:
                    scenario_name = getattr(result, 'scenario_name', 'unknown')
                    stress_data.append({
                        'Сценарий': scenario_name,
                        'Рейтинг': result.rating.value,
                        'PD': f"{result.pd:.4f}"
                    })
                stress_df = pd.DataFrame(stress_data)
                display(stress_df)
                
                # Устойчивость
                if analysis.get('resilience_analysis'):
                    resilience = analysis['resilience_analysis']
                    display(Markdown(f"**Устойчивость:** {resilience['resilience_score']:.2%}"))
        
        with status_output:
            status_output.clear_output()
            print("✅ Полный анализ завершен")
    
    except Exception as e:
        with status_output:
            status_output.clear_output()
            print(f"❌ Ошибка: {e}")
        import traceback
        traceback.print_exc()

# Привязываем обработчики
calculate_base_button.on_click(calculate_base_rating_func)
calculate_forecast_button.on_click(calculate_forecast_rating_func)
calculate_stress_button.on_click(calculate_stress_rating_func)
full_analysis_button.on_click(full_analysis_func)

## 2️⃣ Просмотр конфигурации рейтинга

In [ ]:
# Просмотр конфигурации рейтинга
import yaml

config_path = ROOT / "configs" / "rating_config.yaml"

if config_path.exists():
    with config_path.open("r", encoding="utf-8") as f:
        config = yaml.safe_load(f)
    
    display(Markdown("### 📋 Доступные методологии:"))
    for methodology_key, methodology_config in config.get("methodologies", {}).items():
        name = methodology_config.get("name", methodology_key)
        description = methodology_config.get("description", "")
        display(Markdown(f"- **{methodology_key.upper()}**: {name}"))
        if description:
            display(Markdown(f"  - {description}"))
    
    if config.get("default_probability"):
        display(Markdown("### 📊 Базовая PD по рейтингам:"))
        pd_data = []
        for rating, pd_value in config["default_probability"]["base_pd"].items():
            pd_data.append({"Рейтинг": rating, "PD": f"{pd_value:.4f}"})
        pd_df = pd.DataFrame(pd_data)
        display(pd_df)
else:
    display(Markdown("⚠️ Файл конфигурации не найден: `configs/rating_config.yaml`"))

## 3️⃣ Детальный анализ результатов

После расчета рейтинга здесь можно просмотреть детальные результаты.

In [ ]:
# Функция для детального анализа результатов
def show_detailed_rating_results(result_type: str):
    """Показать детальные результаты рейтинга"""
    if result_type not in rating_results:
        print(f"⚠️ Результаты типа '{result_type}' не найдены")
        return
    
    result = rating_results[result_type]
    
    if result_type == 'base':
        display(Markdown(f"## 📊 Детальные результаты: Base Rating"))
        display(Markdown(f"**Год:** {result.year}"))
        display(Markdown(f"**Рейтинг:** {result.rating.value}"))
        display(Markdown(f"**Score:** {result.score:.2f}"))
        display(Markdown(f"**PD:** {result.pd:.4f}"))
        
        # Детальные метрики
        metrics = result.metrics
        display(Markdown("### Profitability Metrics"))
        display(pd.DataFrame([metrics.profitability.to_dict()]))
        
        display(Markdown("### Leverage Metrics"))
        display(pd.DataFrame([metrics.leverage.to_dict()]))
        
        display(Markdown("### Coverage Metrics"))
        display(pd.DataFrame([metrics.coverage.to_dict()]))
        
        display(Markdown("### Liquidity Metrics"))
        display(pd.DataFrame([metrics.liquidity.to_dict()]))
    
    elif result_type == 'forecast':
        display(Markdown(f"## 📈 Детальные результаты: Forecast Rating"))
        for result in result:
            display(Markdown(f"### Год {result.year}"))
            display(Markdown(f"**Рейтинг:** {result.rating.value}"))
            display(Markdown(f"**PD:** {result.pd:.4f}"))
    
    elif result_type == 'stress':
        display(Markdown(f"## 🧪 Детальные результаты: Stress Rating"))
        for result in result:
            scenario_name = getattr(result, 'scenario_name', 'unknown')
            display(Markdown(f"### Сценарий: {scenario_name}"))
            display(Markdown(f"**Рейтинг:** {result.rating.value}"))
            display(Markdown(f"**PD:** {result.pd:.4f}"))

# Примеры использования:
# show_detailed_rating_results('base')
# show_detailed_rating_results('forecast')
# show_detailed_rating_results('stress')